In [1]:
#__ Import: Numerical __
import numpy as np
from scipy.stats import gaussian_kde

#__ Import: Topophone2 __
from topophone2.utils.read import read_txt
from topophone2.inversion import MarginalStats
from topophone2.utils.math import density_params, harmonic2D_density, density_function

#__ Import: Plotting __
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.tri as tri
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import cmcrameri.cm as cmc

#__ Set up: Plotting __
theme = ['#DC4424', '#74A466', '#24448C', '#F8BC5C']
small, large = 8, 10
plt.rcParams["font.sans-serif"] = ['Arial']
cmap0 = LinearSegmentedColormap.from_list("cmap0", ["white", theme[0]])

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


## Synthetic data

In [11]:
#__ Read array geometry __
out = read_txt(f"./data/TRIAD_15DB/TRIAD_FULL_15DB_topophone2.txt")
x   = out[1]  # Receiver position [m] (receiver, direction)

#__ Load synthetic parameters __
sim_data = np.load('./data/SYNTHETICS.npz')

#__ Unpack: Signal __
slowness_T = sim_data['slowness_T']  # Horizontal slowness    [s.m-1]     (source, mode, direction)
energy_T   = sim_data['energy_T']    # Recorded energy        [uPa2.Hz-1] (source, mode)
xS_T       = sim_data['xS_T']        # Source location        [m]         (source, direction)
w_energy   = sim_data['w_energy']    # Relative source energy [uPa2.Hz-1] (source)

#__ Unpack: Noise __
slowness_J = sim_data['slowness_J']  # Horizontal slowness    [s.m-1]     (source, direction)
energy_J   = sim_data['energy_J']    # Recorded energy        [uPa2.Hz-1] (source)
az_J       = sim_data['az']          # Source azimuth         [deg]       (source)
s0_J       = sim_data['s0'][0]       # Noise slowness         [s.m-1]

#__ Unpack: Sound speed __
z   = sim_data['z']    # Depth [m] (depth,)
c_z = sim_data['c_z']  # Sound speed [m.s-1] (depth,)
sed_thick = 800.0
z_sed = np.linspace(z[-1], z[-1] + sed_thick, 9)
c_sed = np.linspace(1500., 2200., 9)
z_sed = np.append(z_sed, 10e3)
c_sed = np.append(c_sed, c_sed[-1])
z_max = 4.5e3  # Maximum depth [m]
zr    = 1.37e3  # Receiver depth [m]

#__ Estimate energy density __
slowness_T = slowness_T.reshape(-1, 2)
energy_T   = energy_T.reshape(-1)
good = (
    np.all(np.isfinite(slowness_T), axis=1)
    & np.isfinite(energy_T)
    & (energy_T > 1e-12*energy_T.max())
)
s_good = slowness_T[good]
w_good = energy_T[good]
sx, sy = s_good[:,0], s_good[:,1]
pad = 0.15 * max(np.ptp(sx), np.ptp(sy))
Ns  = 200
gx, gy = np.meshgrid(
    np.linspace(sx.min()-pad, sx.max()+pad, Ns),
    np.linspace(sy.min()-pad, sy.max()+pad, Ns),
    indexing='ij'
)
kde     = gaussian_kde(s_good.T, weights=w_good)
w_density = kde(np.vstack([gx.ravel(), gy.ravel()])).reshape(Ns, Ns) * w_good.sum()
w_density /= w_density.max()

#__ Unpack: True model __
m_true = sim_data['m_true']

#__ Read: Inversion results __
m     = np.load('./models/TRIAD_15DB/inversion_results.npz')['m']
w_T   = m[:,0]
xi    = m[:,1:3]
sigma = m[:,3:5]
corr  = m[:,5]
w_J   = m[:,6]
a     = m[:,7]
b     = m[:,8]

#__ Calculate SNR cumulative distribution __
snr_stats = MarginalStats((w_T-w_J)[:,None])
pcum_snr, pcum_snr_vec = snr_stats.probability_density(i=0, cumulative='reverse')

#__ Calculate backazimuth distribution __
_, _, xi_circ, _, _ = density_params(xi, sigma, corr[:,None])
stats_xi = MarginalStats(xi_circ)
p_az, p_az_vec = stats_xi.probability_density(i=1)

#__ Calculate harmonic coefficient distributions __
stats = MarginalStats(m)
p_a, p_a_vec = stats.probability_density(i=7)
p_b, p_b_vec = stats.probability_density(i=8)

#__ Information gain __
ig = np.load('./data/figures/info-gain.npz')
rel_IG_rec = ig['rel_IG_rec']
rel_IG_noisy = ig['rel_IG_noisy']

#__ Noise field __
baz = np.linspace(0, 360, 360)
rho = harmonic2D_density(baz, a[:,None], b[:,None])
rho_true = harmonic2D_density(baz, m_true[7][None,None], m_true[8][None,None])

#__ Calculate density functions __
density = density_function((gx,gy), xi[:1000], sigma[:1000], corr[:1000,None])
density_true = density_function((gx,gy), m_true[1:3][None,:], m_true[3:5][None,:], m_true[5][None,None])
density /= np.max(density, axis=(1,2), keepdims=True)
density_true /= np.max(density_true, axis=(1,2), keepdims=True)

In [ ]:
#________ CREATE FIGURE ________#

#__ Figure __
fig = plt.figure(
    figsize   = (6.5, 4.0),
    facecolor = 'white',
    layout    = 'constrained'
)

#__ Axes __
gs  = GridSpec(
    figure = fig,
    nrows  = 3,
    ncols  = 6
)
axs = [
    fig.add_subplot(gs[0,0:2]),
    fig.add_subplot(gs[0,2:4], aspect='equal'),
    fig.add_subplot(gs[0,4:6], aspect='equal'),
    fig.add_subplot(gs[1,0:2]),
    fig.add_subplot(gs[1,2:4]),#, projection='polar'),
    fig.add_subplot(gs[1,4:6], aspect='equal'),
    fig.add_subplot(gs[2,0]),
    fig.add_subplot(gs[2,1]),
    fig.add_subplot(gs[2,2]),
    fig.add_subplot(gs[2,3:6])
]

#__ Create inset axes __
ax_in = axs[1].inset_axes([0.02, 0.02, 0.30, 0.30], zorder=5)
ax_in.set_facecolor('white')
ax_in.set_aspect('equal')

#__ Labels __
for ax, xlab, ylab in zip(axs,
    [r"Sound speed [km$\,$s$^{-1}$]", r"Easting [km]", r"Slowness (east) [s$\,$km$^{-1}$]", r"SNR threshold [dB]", r"Backazimuth [$\degree$]", r"Slowness (east) [s$\,$km$^{-1}$]", r"$a_1$", r"$b_1$", r"Backazimuth [$\degree$]", r"Information gain [%]"],
    [r"Depth [m]", r"Northing [km]", r"Slowness (north) [s$\,$km$^{-1}$]", r"Probability [%]", r"Energy density [Pa$^2$$\,$Hz$^{-1}$$\,$/$\,\degree$]", r"Slowness (north) [s$\,$km$^{-1}$]", None, None, None, None]
):
    if (xlab is not None): ax.set_xlabel(xlab, fontsize=small)
    if (ylab is not None): ax.set_ylabel(ylab, fontsize=small)

#__ Ticks __
for ax, xlims, ylims, xticks, yticks in zip(axs,
    [(1.4,2.3), (-125,1125), (-0.49,-0.41), (13,17), (0,360), (-0.49,-0.41), None],
    [(6.0,0.0), (-500,500), (0.08,0.16), (0,105), (0,0.2), (0.08,0.16), (0,2)],
    [(0.4,0.1), (300,100), (0.02,0.005), (2,0.5), (90,30), (0.02,0.005), None],
    [(1.0,0.25), (300,100), (0.02,0.005), (50,10), (0.1,0.02), (0.02,0.005), None]
):
    if (xlims is not None): ax.set_xlim(*xlims)
    if (ylims is not None): ax.set_ylim(*ylims)
    if (xticks is not None):
        ax.xaxis.set_major_locator(ticker.MultipleLocator(xticks[0]))
        ax.xaxis.set_minor_locator(ticker.MultipleLocator(xticks[1]))
    else: ax.set_xticks([])
    if (yticks is not None):
        ax.yaxis.set_major_locator(ticker.MultipleLocator(yticks[0]))
        ax.yaxis.set_minor_locator(ticker.MultipleLocator(yticks[1]))
    else: ax.set_yticks([])
    ax.tick_params(labelsize=small)
    ax.grid(color='gainsboro', linewidth=0.5)

#__ Ticks (inset) __
ax_in.set_xlim(-1.5,1.5)
ax_in.set_ylim(-1.5,1.5)
ax_in.set_xticks([0])
ax_in.set_yticks([0])
ax_in.tick_params(
    which='both', direction='in',
    top=True, right=True,
    labelbottom=False, labelleft=False
)
ax_in.grid(color='gainsboro', linewidth=0.5)

#__ Polar axis __
# axs[4].set_theta_zero_location('N')
# axs[4].set_theta_direction(-1)
# axs[4].tick_params(labelbottom=False, labelleft=False)

#________ PLOT: SOUND-SPEED PROFILE ________#

#__ Water column __
axs[0].fill_betweenx(
    1e-3*z, 0.0, 1e-3*c_z,
    facecolor='none', edgecolor='grey',
    hatch='---', linewidth=0.0, alpha=0.3, zorder=0
)
axs[0].plot(1e-3*c_z, 1e-3*z, color='k', linewidth=2.0, zorder=2, label="Water")

#__ Sediment column __
axs[0].fill_betweenx(
    1e-3*z_sed, 0.0, 1e-3*c_sed,
    facecolor='none', edgecolor='k',
    hatch='ooo', linewidth=0.0, alpha=0.3, zorder=0
)
axs[0].plot(1e-3*c_sed, 1e-3*z_sed, color='grey', lw=2.0, zorder=2, label="Sediment")

#__ Receiver/source depths __
axs[0].axhline(1e-3*zr, ls='--', color=theme[1], lw=1.0, zorder=1)
axs[0].axhline(1e-3*z_max, ls='--', color=theme[0], lw=1.0, zorder=2)

#__ Legend __
# axs[0].legend(fontsize=small, ncols=1, frameon=False, loc='right', handlelength=1.5)

#________ PLOT: SOURCE-RECEIVER GEOMETRY ________#

#__ Plot source energy __
axs[1].scatter(
    1e-3*xS_T[:,0], 1e-3*xS_T[:,1],
    marker='.', c=theme[0], s=0.5, alpha=w_energy/w_energy.max(),
    zorder=2
)

#__ Plot array location __
axs[1].scatter(
    0, 0,
    c=theme[1], s=20, marker='^',
    zorder=2
)

#__ Plot receivers __
ax_in.scatter(
    1e-3*x[:,0], 1e-3*x[:,1],
    c=theme[1], s=5, marker='^',
    zorder=1
)

#__ Plot reference line __
axs[1].axline(
    (0,0), (np.sin(np.radians(105)), np.cos(np.radians(105))),
    ls=':', color='gainsboro', lw=1.0,
    zorder=1
)

#________ REAL SIGNAL ________#
axs[2].contourf(
    1e3*gx, 1e3*gy, w_density,
    levels=np.arange(0.2, 1.2, 0.2),
    cmap=cmap0,
    zorder=1
)

#__ Plot reference line __
axs[2].axline(
    (0,0), (np.sin(np.radians(105)), np.cos(np.radians(105))),
    ls=':', color='gainsboro', lw=1.0,
    zorder=1
)

#__ First/second moments __
axs[2].contour(
    1e3*gx, 1e3*gy, density_true[0], levels=[0.5],
    colors='k', linewidths=1.5, linestyles='--',
    zorder=2
)

#________ NOISE FIELD ________#

#__ True model __
axs[4].plot(
    baz, 1e-6*rho_true[0] * 10**(75/10) * np.pi/180,
    color='k', lw=1.5, ls='--',
    zorder=2
)

#__ Estimates __
for i in range(1000):
    if np.any(rho[i] < 0): continue
    axs[4].plot(
        baz, 1e-6*rho[i] * 10**(m[i,6]/10) * np.pi/180,
        color=theme[2], lw=0.3, alpha=0.05,
        zorder=1
    )

#________ SIGNAL FIELD ________#

#__ Plot reference line __
axs[5].axline(
    (0,0), (np.sin(np.radians(105)), np.cos(np.radians(105))),
    ls=':', color='gainsboro', lw=1.0,
    zorder=1
)

#__ True __
axs[5].contour(
    1e3*gx, 1e3*gy, density_true[0], levels=[0.5],
    colors='k', linewidths=1.5, linestyles='--',
    zorder=2
)

for i in range(1000):
    axs[5].contour(
        1e3*gx, 1e3*gy, density[i], levels=[0.5],
        colors=theme[0], linewidths=0.3, alpha=0.05,
        zorder=1
    )

#________ PROBABILITY DENSITY FUNCTIONS ________#

#__ SNR __
axs[3].plot(
    pcum_snr_vec, 100*pcum_snr,
    color=theme[3], lw=2.0,
    zorder=1
)
axs[3].axvline(15, ls='--', color='k', lw=1.5, zorder=2)

#__ Cosine coefficient __
axs[6].plot(
    p_a_vec, p_a,
    color=theme[2], lw=2.0,
    zorder=1
)
axs[6].axvline(m_true[7], ls='--', color='k', lw=1.5, zorder=2)

#__ Sine coeffcient __
axs[7].plot(
    p_b_vec, p_b,
    color=theme[2], lw=2.0,
    zorder=1
)
axs[7].axvline(m_true[8], ls='--', color='k', lw=1.5, zorder=2)

#__ Backazimuth __
axs[8].plot(
    (p_az_vec-180)%360, p_az,
    color=theme[0], lw=2.0,
    zorder=1
)
axs[8].axvline(105, ls='--', color='k', lw=1.5, zorder=2)

#________ INFORMATION GAIN ________#

#__ Labels __
y      = np.arange(4)[::-1]
labels = [r"$\mathbf{\xi}$", r"$\Omega$", r"$\left(a_1, b_1\right)$", r"SNR"]
axs[9].set_yticks(y, labels)

#__ Extra receiver __
axs[9].barh(
    y, rel_IG_rec,
    color=theme[1], label="Extra receiver",
    zorder=1
)

#__ Noise increase __
axs[9].barh(
    y, rel_IG_noisy,
    color=theme[2], label="Increased noise",
    zorder=1)

#__ Plot neutral __
axs[9].axvline(0, c='k', ls='--', lw=1.0, zorder=2)

#__ Legend __
axs[9].legend(fontsize=small, loc='upper right')

#__ Show figure __
fig.show()

## Real data

## Supplementary